In [1]:
import numpy as np
from IPython.display import Image, display
from sedona.spark import SedonaContext
import itertools
import os
import pandas as pd
import geopandas as gpd
from sedona.stats.clustering.dbscan import dbscan

In [2]:
additional_packages = [
    'org.apache.sedona:sedona-spark-3.5_2.12:1.7.2',
    'org.datasyslab:geotools-wrapper:1.7.2-28.5',
]

config_params = {
    "spark.jars.packages": ",".join(additional_packages),
    "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
}

if os.environ.get("SEDONA_COPY_MINIO") == "true":
    config_params = {
        **config_params,
        **{
            "spark.hadoop.fs.s3a.access.key": "sedona",
            "spark.hadoop.fs.s3a.secret.key": "sedona_password",
            "spark.hadoop.fs.s3a.endpoint": "http://minio:9000",
            "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
            "spark.hadoop.fs.s3a.path.style.access": "true",
            "spark.driver.memory": "2G",
            "spark.executor.memory": "2G"
        }
    }

bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

for key, value in config_params.items():
    config = config.config(key, value)

sedona = SedonaContext.create(config.getOrCreate())
sedona.sparkContext.setLogLevel("ERROR")

sedona.sparkContext.setCheckpointDir("checkpoint")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cb8b5cf2-f6f2-43c0-b56e-65e840af97a9;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-3.5_2.12;1.7.2 in central
	found org.apache.sedona#sedona-common;1.7.2 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.wololo#jts2geojson;0.16.1 in central
	found org.locationtech.spatial4j#spatial4j;0.8 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-

In [12]:
df = sedona.read.format("csv").\
    option("header", "true").\
    load(f"s3a://{bucket_name}/source_data/air_quality_moran").\
    where("Date = '12/05/2024'").\
    selectExpr(
        "monotonically_increasing_id() AS id",
        "ST_MakePoint(CAST(`Site Longitude` AS Double), CAST(`Site Latitude` AS double)) AS geom",
        "`Daily AQI Value` AS value"
    )

In [13]:
from sedona.stats.weighting import add_distance_band_column

In [16]:
weights_df = add_distance_band_column(
    dataframe=df,
    threshold=1.0,
    include_self=True
)

In [18]:
# wait until the code will be merged

In [ ]:
moran_result = get_moran_i(weights)